In [1]:
# -*- coding: utf-8 -*-
# 스키마 정합(MART 표준화) 스크립트
# 입력:
#   - 읍면동 최종 이용량_통합.xlsx (시트: 중복합산 우선, 없으면 원본전개)
#   - 관광지수_매핑결과.csv
#   - store_scores.csv
# 출력(엑셀만):
#   - SCHEMA_USAGE_ALIGNED.xlsx  (시트: EMD_HOURLY)
#   - SCHEMA_POI_ALIGNED.xlsx    (시트: POI_MART)
#   - SCHEMA_STORE_ALIGNED.xlsx  (시트: STORE_BASE, STORE_AGG_EMD)

import os, re
import numpy as np
import pandas as pd

# ===== 설정 =====
IN_USAGE_XLSX = r"C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\대중교통이용객\읍면동 최종 이용량_통합.xlsx"   # (시트: 중복합산/원본전개)
IN_POI_CSV    = r"C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\지역\관광지수_매핑결과.csv"
IN_STORE_CSV  = r"C:\Users\hyunj\Seoul_Strolling_Adventure\크롤링\전국_그룹분리_csv\_ckpt\store_scores.csv"

OUT_USAGE_XLSX = "SCHEMA_USAGE_ALIGNED.xlsx"
OUT_POI_XLSX   = "SCHEMA_POI_ALIGNED.xlsx"
OUT_STORE_XLSX = "SCHEMA_STORE_ALIGNED.xlsx"

PREF_SHEET, FALLBACK_SHEET = "중복합산", "원본전개"
HOUR_LABELS = [f"{h:02d}시" for h in range(24)]

# 시도 정규화(별칭 → 공식명)
SIDO_MAP = {
    "서울": "서울특별시", "서울시":"서울특별시", "서울특별시":"서울특별시",
    "부산":"부산광역시","부산시":"부산광역시","부산광역시":"부산광역시",
    "대구":"대구광역시","대구시":"대구광역시","대구광역시":"대구광역시",
    "인천":"인천광역시","인천시":"인천광역시","인천광역시":"인천광역시",
    "광주":"광주광역시","광주광역시":"광주광역시",
    "대전":"대전광역시","대전광역시":"대전광역시",
    "울산":"울산광역시","울산광역시":"울산광역시",
    "세종":"세종특별자치시","세종시":"세종특별자치시","세종특별자치시":"세종특별자치시",
    "경기":"경기도","경기도":"경기도",
    "강원":"강원특별자치도","강원도":"강원특별자치도","강원특별자치도":"강원특별자치도",
    "충북":"충청북도","충청북도":"충청북도",
    "충남":"충청남도","충청남도":"충청남도",
    "전북":"전라북도","전라북도":"전라북도",
    "전남":"전라남도","전라남도":"전라남도",
    "경북":"경상북도","경상북도":"경상북도",
    "경남":"경상남도","경상남도":"경상남도",
    "제주":"제주특별자치도","제주특별자치도":"제주특별자치도",
}

# ===== 공통 유틸 =====
def read_csv_any(path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig","cp949","euc-kr","utf-8"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    # 마지막 시도(인코딩 자동)
    return pd.read_csv(path, encoding_errors="ignore")

def norm_sido(val: str) -> str:
    if pd.isna(val): return val
    v = str(val).strip()
    return SIDO_MAP.get(v, v)

def norm_sigungu(val: str) -> str:
    if pd.isna(val): return val
    v = str(val).strip()
    # "~시/~군/~구" 토막만 남기기(앞뒤 공백/코멘트 제거)
    m = re.search(r"([가-힣0-9·\-]+?(시|군|구))", v)
    return m.group(1) if m else v

def norm_emd(val: str) -> str:
    if pd.isna(val): return val
    v = str(val).strip()
    m = re.search(r"([가-힣0-9·\-]+?(동|읍|면))", v)
    return m.group(1) if m else v

def norm_time_label(t: str) -> str:
    if pd.isna(t): return t
    s = str(t)
    m = re.search(r"(\d{1,2})", s)
    if not m: return s
    h = int(m.group(1)) % 24
    return f"{h:02d}시"

def one_hot_from_theme(series: pd.Series, prefix="theme_") -> pd.DataFrame:
    """ '음식,역사' 형태 또는 리스트/세트 형태를 원-핫 다중칸으로 """
    cats = set()
    cleaned = []
    for v in series.fillna(""):
        if isinstance(v, (list, set, tuple)):
            items = [str(x).strip() for x in v if str(x).strip()]
        else:
            items = [s.strip() for s in str(v).replace("/",",").split(",") if s.strip()]
        cleaned.append(items)
        cats.update(items)
    cats = sorted([c for c in cats if c])[:50]  # 너무 많으면 상한
    data = {f"{prefix}{c}":[1 if c in row else 0 for row in cleaned] for c in cats}
    return pd.DataFrame(data)

# ===== 1) EMD_HOURLY (이용량) 표준화 =====
def build_usage_mart(in_xlsx: str) -> pd.DataFrame:
    try:
        df = pd.read_excel(in_xlsx, sheet_name=PREF_SHEET, engine="openpyxl")
    except Exception:
        df = pd.read_excel(in_xlsx, sheet_name=FALLBACK_SHEET, engine="openpyxl")

    # 필수 컬럼 확보
    need = {"시도","시군구","읍면동","년도","시간대"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"[USAGE] 필수 컬럼 누락: {sorted(miss)}")

    # 정규화
    df["시도"]   = df["시도"].map(norm_sido)
    df["시군구"] = df["시군구"].map(norm_sigungu)
    df["읍면동"] = df["읍면동"].map(norm_emd)
    df["년도"]   = pd.to_numeric(df["년도"], errors="coerce").astype("Int64")
    df["시간대"] = df["시간대"].map(norm_time_label)

    # 이용량 생성/보정
    if "이용량" not in df.columns:
        occ = pd.to_numeric(df.get("발생량", 0), errors="coerce").fillna(0)
        arr = pd.to_numeric(df.get("도착량", 0), errors="coerce").fillna(0)
        df["이용량"] = (occ + arr).astype(int)
    else:
        df["이용량"] = pd.to_numeric(df["이용량"], errors="coerce").fillna(0).astype(int)

    # 혼잡도(있으면 보정)
    if "혼잡도" in df.columns:
        df["혼잡도"] = pd.to_numeric(df["혼잡도"], errors="coerce").clip(0,100).round(2)
    else:
        df["혼잡도"] = np.nan  # 이후 계산 단계에서 채움 가능

    # final_level(있으면 보정)
    if "final_level" in df.columns:
        df["final_level"] = df["final_level"].astype(str)
    else:
        df["final_level"] = pd.NA

    # 시간대 정렬
    cat = pd.Categorical(df["시간대"], categories=HOUR_LABELS, ordered=True)
    df = df.assign(시간대=cat).sort_values(["시도","시군구","읍면동","년도","시간대"]).reset_index(drop=True)

    # 표준 스키마
    cols = ["시도","시군구","읍면동","년도","시간대","이용량","혼잡도","final_level"]
    return df[cols]

# ===== 2) POI_MART (관광지) 표준화 =====
def build_poi_mart(in_csv: str) -> pd.DataFrame:
    raw = read_csv_any(in_csv)

    # 후보 컬럼명 추정
    # 필수 목표: poi_id, 시도, 시군구, 읍면동, lat, lon, 관광지수, 테마*
    cols = list(raw.columns)
    colmap = {}

    # poi_id
    for c in ["poi_id","POI_ID","id","poiId","POI"]:
        if c in cols: colmap["poi_id"] = c; break
    if "poi_id" not in colmap:
        # 없으면 이름/주소로 surrogate id 생성
        if "name" in cols: colmap["name"] = "name"
        elif "명칭" in cols: colmap["name"] = "명칭"
        elif "관광지명" in cols: colmap["name"] = "관광지명"

    # 지역
    for k, cands in {
        "시도":   ["시도","광역시","sido","SIDO","region","광역"],
        "시군구": ["시군구","시구군","sigungu","SIGUNGU","sgg"],
        "읍면동": ["읍면동","법정동","emd","EMD","dong"]
    }.items():
        for c in cands:
            if c in cols: colmap[k] = c; break

    # 좌표
    for k, cands in {"lat":["lat","위도","LAT","y"], "lon":["lon","경도","LON","x"]}.items():
        for c in cands:
            if c in cols: colmap[k] = c; break

    # 관광지수
    for c in ["관광지수","tourism_score","tour_score","score","매력도","관광점수"]:
        if c in cols: colmap["관광지수"] = c; break

    # 테마
    theme_col = None
    for c in ["테마","theme","themes","category","카테고리","분류"]:
        if c in cols: theme_col = c; break

    df = raw.copy()

    # 최소 키 생성
    if "poi_id" in colmap:
        df["poi_id"] = df[colmap["poi_id"]]
    else:
        # surrogate
        df["poi_id"] = (df.get(colmap.get("name"), pd.Series(range(len(df))))).astype(str)

    # 지역 정규화
    for k in ["시도","시군구","읍면동"]:
        src = colmap.get(k)
        if src:
            if k == "시도":
                df[k] = df[src].map(norm_sido)
            elif k == "시군구":
                df[k] = df[src].map(norm_sigungu)
            else:
                df[k] = df[src].map(norm_emd)
        else:
            df[k] = pd.NA

    # 좌표
    df["lat"] = pd.to_numeric(df.get(colmap.get("lat", ""), np.nan), errors="coerce")
    df["lon"] = pd.to_numeric(df.get(colmap.get("lon", ""), np.nan), errors="coerce")

    # 관광지수
    df["관광지수"] = pd.to_numeric(df.get(colmap.get("관광지수",""), np.nan), errors="coerce")

    # 테마 원-핫
    if theme_col:
        oh = one_hot_from_theme(df[theme_col], prefix="theme_")
        df = pd.concat([df, oh], axis=1)

    # 표준 스키마
    base_cols = ["poi_id","시도","시군구","읍면동","lat","lon","관광지수"]
    theme_cols = [c for c in df.columns if c.startswith("theme_")]
    keep = base_cols + theme_cols
    return df[keep].copy()

# ===== 3) STORE_BASE (리뷰/인기도) 표준화 & EMD 집계 =====
def parse_sido_from_region_or_addr(region: str, address: str) -> str:
    if pd.notna(region) and str(region).strip():
        return norm_sido(str(region).strip())
    if pd.isna(address): return pd.NA
    # 주소 앞 토큰에서 시도 추출
    s = str(address).strip()
    # 긴 이름 먼저 매칭
    for name in ["세종특별자치시","제주특별자치도","강원특별자치도","서울특별시","부산광역시","대구광역시",
                 "인천광역시","광주광역시","대전광역시","울산광역시",
                 "경기도","충청북도","충청남도","전라북도","전라남도","경상북도","경상남도"]:
        if name in s: return name
    # 약칭
    for alias, full in SIDO_MAP.items():
        if alias in s: return full
    return pd.NA

def extract_sigungu(addr: str) -> str:
    if pd.isna(addr): return pd.NA
    m = re.search(r"([가-힣0-9·\-]+?(시|군|구))", str(addr))
    return m.group(1) if m else pd.NA

def extract_emd(addr: str) -> str:
    if pd.isna(addr): return pd.NA
    m = re.search(r"([가-힣0-9·\-]+?(동|읍|면))", str(addr))
    return m.group(1) if m else pd.NA

def build_store_mart(in_csv: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = read_csv_any(in_csv).copy()
    cols = list(raw.columns)

    # 표준 컬럼 매핑 추정
    col_region = None
    for c in ["region","시도","SIDO","sido"]:
        if c in cols: col_region = c; break

    col_name    = next((c for c in ["store_name","name","매장명"] if c in cols), None)
    col_addr    = next((c for c in ["address","addr","주소"] if c in cols), None)
    col_score   = next((c for c in ["review_score","score","인기도점수"] if c in cols), None)
    col_count   = next((c for c in ["count","리뷰수","review_count"] if c in cols), None)
    col_sent    = next((c for c in ["avg_sentiment","sentiment","평균감성"] if c in cols), None)
    col_prob    = next((c for c in ["avg_class_prob","class_prob","평균확신도"] if c in cols), None)

    df = pd.DataFrame()
    df["시도"] = raw.apply(lambda r: parse_sido_from_region_or_addr(r.get(col_region), r.get(col_addr)), axis=1)
    df["시군구"] = raw[col_addr].map(extract_sigungu) if col_addr else pd.NA
    df["읍면동"] = raw[col_addr].map(extract_emd) if col_addr else pd.NA

    if col_name:  df["store_name"] = raw[col_name]
    if col_addr:  df["address"]    = raw[col_addr]

    df["review_score"] = pd.to_numeric(raw.get(col_score, np.nan), errors="coerce")
    df["count"]        = pd.to_numeric(raw.get(col_count, np.nan), errors="coerce")
    df["avg_sentiment"]= pd.to_numeric(raw.get(col_sent,  np.nan), errors="coerce")
    df["avg_class_prob"]=pd.to_numeric(raw.get(col_prob,  np.nan), errors="coerce")

    # 기본 클린
    df["시도"]   = df["시도"].map(norm_sido)
    df["시군구"] = df["시군구"].map(norm_sigungu)
    df["읍면동"] = df["읍면동"].map(norm_emd)

    # STORE_BASE
    store_base = df.copy()

    # EMD 집계(인기도/리뷰 요약) — 이후 POI 백오프용
    agg = (store_base
           .groupby(["시도","시군구","읍면동"], dropna=False)
           .agg(
               review_score_mean=("review_score","mean"),
               review_score_median=("review_score","median"),
               review_count_sum=("count","sum"),
               place_cnt=("store_name","count")
           )
           .reset_index())

    return store_base, agg

# ===== 실행 =====
if __name__ == "__main__":
    # 1) EMD_HOURLY
    emd_hourly = build_usage_mart(IN_USAGE_XLSX)
    with pd.ExcelWriter(OUT_USAGE_XLSX, engine="xlsxwriter") as w:
        emd_hourly.to_excel(w, sheet_name="EMD_HOURLY", index=False)

    # 2) POI_MART
    poi_mart = build_poi_mart(IN_POI_CSV)
    with pd.ExcelWriter(OUT_POI_XLSX, engine="xlsxwriter") as w:
        poi_mart.to_excel(w, sheet_name="POI_MART", index=False)

    # 3) STORE_BASE & STORE_AGG_EMD
    store_base, store_agg = build_store_mart(IN_STORE_CSV)
    with pd.ExcelWriter(OUT_STORE_XLSX, engine="xlsxwriter") as w:
        store_base.to_excel(w, sheet_name="STORE_BASE", index=False)
        store_agg.to_excel(w, sheet_name="STORE_AGG_EMD", index=False)

    print("[DONE] 스키마 정합 완료:")
    print(" -", OUT_USAGE_XLSX, "(EMD_HOURLY)")
    print(" -", OUT_POI_XLSX,   "(POI_MART)")
    print(" -", OUT_STORE_XLSX, "(STORE_BASE, STORE_AGG_EMD)")

[DONE] 스키마 정합 완료:
 - SCHEMA_USAGE_ALIGNED.xlsx (EMD_HOURLY)
 - SCHEMA_POI_ALIGNED.xlsx (POI_MART)
 - SCHEMA_STORE_ALIGNED.xlsx (STORE_BASE, STORE_AGG_EMD)


In [1]:
import os
os.environ["KAKAO_REST_API_KEY"] = "0be64775eb0d51574480225b2b175243"

In [ ]:
# -*- coding: utf-8 -*-
# 추천 파이프라인 (스키마 정합 → 조인 → 점수화 → 슬롯별 추천 → 엑셀 저장)
# 필요 패키지: pandas numpy openpyxl XlsxWriter
import os, re, math, requests
import numpy as np
import pandas as pd

# =========================
# 설정 (원하면 바꿔서 사용)
# =========================
DATA_DIR = "."
# 입력 파일(스키마 정합 산출물이 있으면 우선 사용)
SCHEMA_USAGE = os.path.join(DATA_DIR, "SCHEMA_USAGE_ALIGNED.xlsx")   # EMD_HOURLY
SCHEMA_POI   = os.path.join(DATA_DIR, "SCHEMA_POI_ALIGNED.xlsx")     # POI_MART
SCHEMA_STORE = os.path.join(DATA_DIR, "SCHEMA_STORE_ALIGNED.xlsx")   # STORE_BASE/STORE_AGG_EMD

RAW_USAGE_XLSX = r"C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\대중교통이용객\읍면동 최종 이용량_통합.xlsx"   # (시트: 중복합산/원본전개)
RAW_POI_CSV    = r"C:\Users\hyunj\Seoul_Strolling_Adventure\dataset\지역\관광지수_매핑결과.csv"
RAW_STORE_CSV  = r"C:\Users\hyunj\Seoul_Strolling_Adventure\크롤링\전국_그룹분리_csv\_ckpt\store_scores.csv"

OUT_XLSX = os.path.join(DATA_DIR, "추천_결과.xlsx")

# ============ 사용자 입력 ============
# [1] 여행지 선택 — Kakao 키워드/주소 검색 쿼리
USER_QUERY   = "강남역"     # 예시: "강남역", "속초 해수욕장", "서울 송파구 잠실동"
TRAVEL_MODE  = "transit"    # "walk" | "transit"
SCHEDULE     = "당일치기"    # "당일치기" | "1박2일" | "2박3일"

# 도보반경(m) — walk 모드에서만 적용
WALK_RADIUS_M = 1000

# [3] 테마 가중치(합은 내부에서 1로 정규화)
THEME_WEIGHTS = {"음식": 10, "역사": 5, "스포츠": 0}

# 점수 가중치(α 관광/β 인기도/γ 테마/δ 혼잡(도착지))
ALPHA, BETA, GAMMA, DELTA = 0.35, 0.25, 0.30, 0.10
# 이동 페널티(거리) — km * LAMBDA_KM
LAMBDA_KM = 0.08
# (신규) 대중교통 혼잡 페널티 — (출발/도착 혼잡 평균) * LAMBDA_CONG
LAMBDA_CONG = 0.35
# 대중교통 평균 속도(휴리스틱, km/h)
AVG_TRANSIT_KMPH = 22.0

# ============ 공통 유틸 ============
HOUR_LABELS = [f"{h:02d}시" for h in range(24)]
SIDO_MAP = {
    "서울":"서울특별시","서울시":"서울특별시","서울특별시":"서울특별시",
    "부산":"부산광역시","부산광역시":"부산광역시",
    "대구":"대구광역시","대구광역시":"대구광역시",
    "인천":"인천광역시","인천광역시":"인천광역시",
    "광주":"광주광역시","광주광역시":"광주광역시",
    "대전":"대전광역시","대전광역시":"대전광역시",
    "울산":"울산광역시","울산광역시":"울산광역시",
    "세종":"세종특별자치시","세종특별자치시":"세종특별자치시",
    "경기":"경기도","경기도":"경기도",
    "강원":"강원특별자치도","강원도":"강원특별자치도","강원특별자치도":"강원특별자치도",
    "충북":"충청북도","충청북도":"충청북도",
    "충남":"충청남도","충청남도":"충청남도",
    "전북":"전라북도","전라북도":"전라북도",
    "전남":"전라남도","전라남도":"전라남도",
    "경북":"경상북도","경상북도":"경상북도",
    "경남":"경상남도","경상남도":"경상남도",
    "제주":"제주특별자치도","제주특별자치도":"제주특별자치도",
}
def norm_sido(v):
    if pd.isna(v): return v
    s=str(v).strip(); return SIDO_MAP.get(s,s)
def norm_sigungu(v):
    if pd.isna(v): return v
    m=re.search(r"([가-힣0-9·\-]+?(시|군|구))",str(v)); return m.group(1) if m else str(v).strip()
def norm_emd(v):
    if pd.isna(v): return v
    m=re.search(r"([가-힣0-9·\-]+?(동|읍|면))",str(v)); return m.group(1) if m else str(v).strip()
def norm_time_label(t):
    if pd.isna(t): return t
    m=re.search(r"(\d{1,2})",str(t)); 
    h=int(m.group(1))%24 if m else 0
    return f"{h:02d}시"
def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(x) for x in [lat1,lon1,lat2,lon2]): return np.inf
    R=6371.0088
    f1,f2=math.radians(lat1), math.radians(lat2)
    dlat=math.radians(lat2-lat1); dlon=math.radians(lon2-lon1)
    a=math.sin(dlat/2)**2+math.cos(f1)*math.cos(f2)*math.sin(dlon/2)**2
    return 2*R*math.asin(math.sqrt(a))

# ============ Kakao Local API ============ 
# 공식 문서: search/address, search/keyword, geo/coord2regioncode (Authorization: KakaoAK {REST_API_KEY})
# https://dapi.kakao.com/v2/local/search/address.json
# https://dapi.kakao.com/v2/local/search/keyword.json
# https://dapi.kakao.com/v2/local/geo/coord2regioncode.json
KAKAO_KEY = os.environ.get("KAKAO_REST_API_KEY")
def _kakao_get(url, params):
    if not KAKAO_KEY:
        raise RuntimeError("환경변수 KAKAO_REST_API_KEY 가 설정되지 않았습니다.")
    headers={"Authorization": f"KakaoAK {KAKAO_KEY}"}
    r=requests.get(url, headers=headers, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def kakao_search_keyword(query, x=None, y=None, radius=None, size=5):
    url="https://dapi.kakao.com/v2/local/search/keyword.json"
    params={"query": query, "size": size}
    if x is not None and y is not None: params.update({"x": x, "y": y})
    if radius is not None: params["radius"]=radius
    j=_kakao_get(url, params)
    docs=j.get("documents",[])
    if not docs: return None
    top=docs[0]
    return {
        "name": top.get("place_name"),
        "address": top.get("road_address_name") or top.get("address_name"),
        "lon": float(top.get("x")), "lat": float(top.get("y"))
    }

def kakao_search_address(query):
    url="https://dapi.kakao.com/v2/local/search/address.json"
    j=_kakao_get(url, {"query": query})
    docs=j.get("documents",[])
    if not docs: return None
    doc=docs[0]
    # address/road_address 중 존재하는 것의 좌표 사용
    x=doc.get("x") or (doc.get("address") or {}).get("x") or (doc.get("road_address") or {}).get("x")
    y=doc.get("y") or (doc.get("address") or {}).get("y") or (doc.get("road_address") or {}).get("y")
    if x is None or y is None: return None
    return {"name": query, "address": query, "lon": float(x), "lat": float(y)}

def kakao_coord2emd(lon, lat):
    url="https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
    j=_kakao_get(url, {"x": lon, "y": lat})
    docs=j.get("documents",[])
    # region_type B = 법정동(읍/면/동)
    b = next((d for d in docs if d.get("region_type")=="B"), None)
    if not b: 
        return None
    return {
        "sido": norm_sido(b.get("region_1depth_name")),
        "sigungu": norm_sigungu(b.get("region_2depth_name")),
        "emd": norm_emd(b.get("region_3depth_name")),
        "code": b.get("code"),
        "lon": float(b.get("x")), "lat": float(b.get("y"))
    }

def resolve_start_from_query(query):
    # 1) 키워드 검색
    hit = kakao_search_keyword(query)
    if not hit:
        # 2) 주소 검색 fallback
        hit = kakao_search_address(query)
    if not hit: 
        raise RuntimeError(f"Kakao에서 '{query}' 결과를 찾지 못했습니다.")
    emd = kakao_coord2emd(hit["lon"], hit["lat"])
    if not emd:
        raise RuntimeError("좌표→법정동 변환 실패")
    out = {**hit, **emd}
    return out  # name,address,lon,lat,sido,sigungu,emd,code

# ============ 데이터 로드(정합물 우선) ============
def read_csv_any(path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig","cp949","euc-kr","utf-8"):
        try: return pd.read_csv(path, encoding=enc)
        except Exception: continue
    return pd.read_csv(path, encoding_errors="ignore")

def load_usage():
    if os.path.exists(SCHEMA_USAGE):
        df=pd.read_excel(SCHEMA_USAGE, sheet_name="EMD_HOURLY", engine="openpyxl")
    else:
        try: df=pd.read_excel(RAW_USAGE_XLSX, sheet_name="중복합산", engine="openpyxl")
        except: df=pd.read_excel(RAW_USAGE_XLSX, sheet_name="원본전개", engine="openpyxl")
        need={"시도","시군구","읍면동","년도","시간대"}
        if not need.issubset(df.columns): 
            raise ValueError("[USAGE] 필수 컬럼 누락")
        df["시도"]=df["시도"].map(norm_sido)
        df["시군구"]=df["시군구"].map(norm_sigungu)
        df["읍면동"]=df["읍면동"].map(norm_emd)
        df["년도"]=pd.to_numeric(df["년도"],errors="coerce").astype("Int64")
        df["시간대"]=df["시간대"].map(norm_time_label)
        if "이용량" not in df.columns:
            occ=pd.to_numeric(df.get("발생량",0),errors="coerce").fillna(0)
            arr=pd.to_numeric(df.get("도착량",0),errors="coerce").fillna(0)
            df["이용량"]=(occ+arr).astype(int)
        if "혼잡도" not in df.columns: df["혼잡도"]=np.nan
        if "final_level" not in df.columns: df["final_level"]=pd.NA
    cat=pd.Categorical(df["시간대"], categories=HOUR_LABELS, ordered=True)
    df=df.assign(시간대=cat).sort_values(["시도","시군구","읍면동","년도","시간대"]).reset_index(drop=True)
    return df[["시도","시군구","읍면동","년도","시간대","이용량","혼잡도","final_level"]].copy()

def one_hot_from_theme(series: pd.Series, prefix="theme_"):
    cats=set(); cleaned=[]
    for v in series.fillna(""):
        if isinstance(v,(list,set,tuple)):
            items=[str(x).strip() for x in v if str(x).strip()]
        else:
            items=[s.strip() for s in str(v).replace("/",",").split(",") if s.strip()]
        cleaned.append(items); cats.update(items)
    cats=sorted([c for c in cats if c])[:50]
    data={f"{prefix}{c}":[1 if c in row else 0 for row in cleaned] for c in cats}
    return pd.DataFrame(data)

def load_poi():
    if os.path.exists(SCHEMA_POI):
        return pd.read_excel(SCHEMA_POI, sheet_name="POI_MART", engine="openpyxl")
    raw=read_csv_any(RAW_POI_CSV)
    cols=list(raw.columns); df=raw.copy()
    def pick(cands): 
        for c in cands:
            if c in cols: return c
        return None
    c_id=pick(["poi_id","POI_ID","id","poiId","POI"])
    c_name=pick(["name","명칭","관광지명"])
    c_sido=pick(["시도","광역시","sido","SIDO","region"])
    c_sgg =pick(["시군구","시구군","sigungu","SIGUNGU","sgg"])
    c_emd =pick(["읍면동","법정동","emd","EMD","dong"])
    c_lat =pick(["lat","위도","LAT","y"])
    c_lon =pick(["lon","경도","LON","x"])
    c_theme=pick(["테마","theme","themes","category","카테고리","분류"])
    c_score=pick(["관광지수","tourism_score","tour_score","score","매력도","관광점수"])

    if c_id: df["poi_id"]=df[c_id].astype(str)
    elif c_name: df["poi_id"]=df[c_name].astype(str)
    else: df["poi_id"]=df.index.astype(str)

    if c_name and "name" not in df: df["name"]=df[c_name]
    elif "name" not in df: df["name"]=df["poi_id"]

    df["시도"]=df[c_sido].map(norm_sido) if c_sido else pd.NA
    df["시군구"]=df[c_sgg].map(norm_sigungu) if c_sgg else pd.NA
    df["읍면동"]=df[c_emd].map(norm_emd) if c_emd else pd.NA
    df["lat"]=pd.to_numeric(df[c_lat],errors="coerce") if c_lat else np.nan
    df["lon"]=pd.to_numeric(df[c_lon],errors="coerce") if c_lon else np.nan
    df["관광지수"]=pd.to_numeric(df[c_score],errors="coerce") if c_score else np.nan

    if c_theme:
        oh=one_hot_from_theme(df[c_theme], prefix="theme_")
        df=pd.concat([df, oh], axis=1)

    base=["poi_id","name","시도","시군구","읍면동","lat","lon","관광지수"]
    themes=[c for c in df.columns if str(c).startswith("theme_")]
    return df[base+themes].copy()

def load_store():
    if os.path.exists(SCHEMA_STORE):
        base=pd.read_excel(SCHEMA_STORE, sheet_name="STORE_BASE", engine="openpyxl")
        agg =pd.read_excel(SCHEMA_STORE, sheet_name="STORE_AGG_EMD", engine="openpyxl")
        return base, agg
    raw=read_csv_any(RAW_STORE_CSV)
    cols=list(raw.columns)
    def pick(cands): 
        for c in cands:
            if c in cols: return c
        return None
    c_region=pick(["region","시도","SIDO","sido"])
    c_name  =pick(["store_name","name","매장명"])
    c_addr  =pick(["address","addr","주소"])
    c_score =pick(["review_score","score","인기도점수"])
    c_cnt   =pick(["count","리뷰수","review_count"])
    c_sent  =pick(["avg_sentiment","sentiment","평균감성"])
    c_prob  =pick(["avg_class_prob","class_prob","평균확신도"])

    df=pd.DataFrame()
    def parse_sido(region, addr):
        if pd.notna(region) and str(region).strip(): return norm_sido(region)
        if pd.isna(addr): return pd.NA
        s=str(addr)
        for name in ["세종특별자치시","제주특별자치도","강원특별자치도","서울특별시","부산광역시","대구광역시",
                     "인천광역시","광주광역시","대전광역시","울산광역시",
                     "경기도","충청북도","충청남도","전라북도","전라남도","경상북도","경상남도"]:
            if name in s: return name
        for alias, full in SIDO_MAP.items():
            if alias in s: return full
        return pd.NA

    df["시도"]=raw.apply(lambda r: parse_sido(r.get(c_region), r.get(c_addr)), axis=1)
    def ex_sgg(addr):
        if pd.isna(addr): return pd.NA
        m=re.search(r"([가-힣0-9·\-]+?(시|군|구))", str(addr)); return m.group(1) if m else pd.NA
    def ex_emd(addr):
        if pd.isna(addr): return pd.NA
        m=re.search(r"([가-힣0-9·\-]+?(동|읍|면))", str(addr)); return m.group(1) if m else pd.NA
    df["시군구"]=raw[c_addr].map(ex_sgg) if c_addr else pd.NA
    df["읍면동"]=raw[c_addr].map(ex_emd) if c_addr else pd.NA

    if c_name: df["store_name"]=raw[c_name]
    if c_addr: df["address"]=raw[c_addr]
    df["review_score"]=pd.to_numeric(raw.get(c_score, np.nan),errors="coerce")
    df["count"]=pd.to_numeric(raw.get(c_cnt, np.nan),errors="coerce")
    df["avg_sentiment"]=pd.to_numeric(raw.get(c_sent, np.nan),errors="coerce")
    df["avg_class_prob"]=pd.to_numeric(raw.get(c_prob, np.nan),errors="coerce")

    df["시도"]=df["시도"].map(norm_sido)
    df["시군구"]=df["시군구"].map(norm_sigungu)
    df["읍면동"]=df["읍면동"].map(norm_emd)

    agg=(df.groupby(["시도","시군구","읍면동"], dropna=False)
           .agg(review_score_mean=("review_score","mean"),
                review_score_median=("review_score","median"),
                review_count_sum=("count","sum"),
                place_cnt=("store_name","count"))
           .reset_index())
    return df, agg

# ============ 정규화 & 슬롯 스코어 ============
def normalize_scores(poi: pd.DataFrame, store_agg: pd.DataFrame):
    # --- 관광지수_norm: 시도 기준 z→0~1 ---
    poi = poi.copy()
    poi["관광지수"] = pd.to_numeric(poi["관광지수"], errors="coerce")

    # 시도별 평균/표준편차
    g = poi.groupby("시도")["관광지수"]
    mu = g.transform("mean")
    sd = g.transform("std").replace(0, np.nan)
    z = (poi["관광지수"] - mu) / sd
    z = z.fillna(0.0)

    # 시도별 z의 min/max로 0~1 스케일
    z_min = z.groupby(poi["시도"]).transform("min")
    z_max = z.groupby(poi["시도"]).transform("max")
    poi["관광지수_norm"] = np.where((z_max - z_min) > 0, (z - z_min) / (z_max - z_min), 0.0)

    # --- 인기도_norm: 시군구 기준 min-max (store_agg의 review_score_mean 사용) ---
    sagg = store_agg.copy()
    sagg["review_score_mean"] = pd.to_numeric(sagg["review_score_mean"], errors="coerce")

    # 시군구별 min/max
    key = ["시도","시군구"]
    g2 = sagg.groupby(key)["review_score_mean"]
    mn = g2.transform("min")
    mx = g2.transform("max")
    sagg["pop_norm"] = np.where((mx - mn) > 0, (sagg["review_score_mean"] - mn) / (mx - mn), 0.0)

    # POI에 EMD 인기도 붙이기
    poi = poi.merge(sagg[["시도","시군구","읍면동","pop_norm"]], 
                    on=["시도","시군구","읍면동"], how="left")

    # 시군구 평균으로 백오프
    sgg_mean = (sagg.groupby(key, as_index=False)["pop_norm"].mean()
                    .rename(columns={"pop_norm":"pop_norm_sgg"}))
    poi = poi.merge(sgg_mean, on=key, how="left")

    poi["인기도_norm"] = poi["pop_norm"].fillna(poi["pop_norm_sgg"]).fillna(0.0)
    poi.drop(columns=[c for c in ["pop_norm","pop_norm_sgg"] if c in poi], inplace=True)

    return poi


def build_theme_weight_vector(poi_cols, user_weights: dict):
    wsum=sum(max(0,float(v)) for v in user_weights.values()) or 1.0
    w={k: max(0,float(v))/wsum for k,v in user_weights.items()}
    mapping={}
    for k,v in w.items():
        col=f"theme_{k}"
        if col in poi_cols: mapping[col]=v
    return mapping

def compute_slot_scores(poi: pd.DataFrame, emd_hourly: pd.DataFrame, slot_hour: str,
                        mode: str, theme_w: dict):
    emd = emd_hourly[emd_hourly["시간대"]==slot_hour].copy()
    if "혼잡도" not in emd.columns or emd["혼잡도"].isna().all():
        def mm(g):
            x=g["이용량"].astype(float).values
            mn, mx = np.min(x), np.max(x)
            g["혼잡도"]= (g["이용량"]-mn)/(mx-mn) if mx>mn else 0.0
            g["혼잡도"] = g["혼잡도"]*100.0
            return g
        emd=emd.groupby(["시도","시간대"], group_keys=False).apply(mm)
    emd["혼잡도_norm"]=pd.to_numeric(emd["혼잡도"], errors="coerce").fillna(0)/100.0
    if emd["혼잡도_norm"].max()>1:
        emd["혼잡도_norm"]=emd["혼잡도_norm"]/emd["혼잡도_norm"].max()

    base = poi.merge(emd[["시도","시군구","읍면동","시간대","혼잡도_norm"]],
                     on=["시도","시군구","읍면동"], how="left")
    base["혼잡도_norm"]=base["혼잡도_norm"].fillna(0.0)

    base["theme_score"]=0.0
    for col, w in theme_w.items():
        if col in base.columns:
            base["theme_score"] += base[col]*w

    if mode=="walk":
        base["final_score"] = (ALPHA*base["관광지수_norm"]
                              +BETA *base["인기도_norm"]
                              +GAMMA*base["theme_score"])
    else:
        base["final_score"] = (ALPHA*base["관광지수_norm"]
                              +BETA *base["인기도_norm"]
                              +GAMMA*base["theme_score"]
                              +DELTA*(1.0 - base["혼잡도_norm"]))
    base["시간대"]=slot_hour
    return base.sort_values("final_score", ascending=False)

# ============ 슬롯/후보/일정 ============ 
def make_slots(schedule: str):
    if schedule=="당일치기":
        return ["10시","12시","14시","16시","18시"]
    if schedule=="1박2일":
        return ["10시","12시","14시","16시","18시","20시",
                "10시","12시","14시","16시"]
    if schedule=="2박3일":
        return ["10시","12시","14시","16시","18시","20시"]*3
    return ["10시","12시","14시","16시","18시"]

def filter_walk_radius(poi: pd.DataFrame, lat, lon, radius_m=1000):
    if pd.isna(lat) or pd.isna(lon): 
        return poi
    dists = poi.apply(lambda r: haversine_km(lat, lon, r.get("lat"), r.get("lon"))*1000, axis=1)
    return poi[dists <= radius_m].copy()

def select_itinerary_transit(candidates_by_slot: dict, emd_hourly: pd.DataFrame,
                             start_info: dict, lambda_km=LAMBDA_KM, lambda_cong=LAMBDA_CONG):
    """
    대중교통 Greedy:
      - 첫 슬롯: 시작점에서 거리/혼잡 페널티 적용한 점수 최대
      - 이후 슬롯: (점수 - λ_km*거리 - λ_cong*출발/도착 혼잡평균) 최대
      - 혼잡도는 각 슬롯의 시간대 기준 (출발=이전 선택, 도착=후보)
    """
    def emd_cong(sido, sgg, emd, hour):
        row = emd_hourly[(emd_hourly["시도"]==sido)&(emd_hourly["시군구"]==sgg)&
                         (emd_hourly["읍면동"]==emd)&(emd_hourly["시간대"]==hour)]
        if row.empty: return 0.0
        val = row.iloc[0].get("혼잡도")
        if pd.isna(val): return 0.0
        v = float(val)/100.0
        return v if 0<=v<=1 else max(0.0, min(1.0, v))

    slots = list(candidates_by_slot.keys())
    itinerary=[]
    prev_lat, prev_lon = start_info["lat"], start_info["lon"]
    prev_sido, prev_sgg, prev_emd = start_info["sido"], start_info["sigungu"], start_info["emd"]

    for slot in slots:
        cand = candidates_by_slot[slot].copy()
        used_ids=set([x["poi_id"] for x in itinerary if x.get("poi_id")])
        cand = cand[~cand["poi_id"].isin(used_ids)]
        if cand.empty:
            itinerary.append({"시간대":slot})
            continue
        cand=cand.copy()
        # 거리/혼잡 페널티
        # 출발 혼잡: 이전 선택의 EMD 기준 (같은 슬롯 시간)
        cong_from = emd_cong(prev_sido, prev_sgg, prev_emd, slot)
        cand["move_km"]=cand.apply(lambda r: haversine_km(prev_lat, prev_lon, r["lat"], r["lon"]), axis=1)
        cong_to = cand["혼잡도_norm"].fillna(0.0)  # 도착 EMD 혼잡(이미 붙어 있음)
        cand["score_adj"]=cand["final_score"] - (lambda_km*cand["move_km"]) - (lambda_cong*((cong_from+cong_to)/2.0))
        pick=cand.sort_values("score_adj", ascending=False).iloc[0]

        itinerary.append({
            "시간대": slot,
            "poi_id": pick.get("poi_id"),
            "name": pick.get("name"),
            "시도": pick.get("시도"), "시군구": pick.get("시군구"), "읍면동": pick.get("읍면동"),
            "lat": pick.get("lat"), "lon": pick.get("lon"),
            "관광지수_norm": float(pick.get("관광지수_norm",0)),
            "인기도_norm": float(pick.get("인기도_norm",0)),
            "theme_score": float(pick.get("theme_score",0)),
            "혼잡도_norm": float(pick.get("혼잡도_norm",0)),
            "final_score": float(pick.get("final_score",0)),
            "이동거리_km": float(pick.get("move_km", np.nan)),
            "score_adj": float(pick.get("score_adj", np.nan))
        })
        # 다음 루프를 위한 prev 업데이트
        prev_lat, prev_lon = pick.get("lat"), pick.get("lon")
        prev_sido, prev_sgg, prev_emd = pick.get("시도"), pick.get("시군구"), pick.get("읍면동")

    return pd.DataFrame(itinerary)

def select_itinerary_walk(candidates_by_slot: dict, start_lat=None, start_lon=None, lambda_km=LAMBDA_KM):
    itinerary=[]
    prev_lat, prev_lon = start_lat, start_lon
    for slot in candidates_by_slot:
        cand = candidates_by_slot[slot].copy()
        used_ids=set([x["poi_id"] for x in itinerary if x.get("poi_id")])
        cand = cand[~cand["poi_id"].isin(used_ids)]
        if cand.empty:
            itinerary.append({"시간대":slot})
            continue
        if prev_lat is None or prev_lon is None or cand[["lat","lon"]].isna().any().all():
            pick=cand.iloc[0]; move_km=None; score_adj=pick["final_score"]
        else:
            cand["move_km"]=cand.apply(lambda r: haversine_km(prev_lat, prev_lon, r["lat"], r["lon"]), axis=1)
            cand["score_adj"]=cand["final_score"] - lambda_km*cand["move_km"]
            pick=cand.sort_values("score_adj", ascending=False).iloc[0]
            move_km=float(pick["move_km"]); score_adj=float(pick["score_adj"])
        itinerary.append({
            "시간대": slot, "poi_id": pick.get("poi_id"), "name": pick.get("name"),
            "시도": pick.get("시도"), "시군구": pick.get("시군구"), "읍면동": pick.get("읍면동"),
            "lat": pick.get("lat"), "lon": pick.get("lon"),
            "관광지수_norm": float(pick.get("관광지수_norm",0)),
            "인기도_norm": float(pick.get("인기도_norm",0)),
            "theme_score": float(pick.get("theme_score",0)),
            "혼잡도_norm": float(pick.get("혼잡도_norm",0)),
            "final_score": float(pick.get("final_score",0)),
            "이동거리_km": move_km, "score_adj": score_adj
        })
        prev_lat, prev_lon = pick.get("lat"), pick.get("lon")
    return pd.DataFrame(itinerary)

# ============ 실행 ============
if __name__=="__main__":
    # 0) Kakao로 시작점/읍면동 결정
    start = resolve_start_from_query(USER_QUERY)  # name,address,lon,lat,sido,sigungu,emd,code
    START_LAT, START_LON = start["lat"], start["lon"]

    # 1) 데이터 로드
    emd_hourly = load_usage()
    poi_mart   = load_poi()
    store_base, store_agg = load_store()

    # 2) 정규화
    poi_scored = normalize_scores(poi_mart.copy(), store_agg)

    # 3) EMD(읍면동) 단위 필터링
    #   - 기본: 시작 EMD와 같은 시군구 내 후보 (읍면동 범위를 너무 타이트하게 잡으면 후보가 적어질 수 있음)
    poi_scored = poi_scored[
        (poi_scored["시도"]==start["sido"]) & (poi_scored["시군구"]==start["sigungu"])
    ].copy()

    # 4) 테마 가중치
    theme_w = build_theme_weight_vector(poi_scored.columns, THEME_WEIGHTS)

    # 5) 슬롯 생성
    slots = make_slots(SCHEDULE)

    # 6) 슬롯별 후보 스코어
    candidates_by_slot = {}
    for slot in slots:
        base = compute_slot_scores(poi_scored, emd_hourly, slot, TRAVEL_MODE, theme_w)
        if TRAVEL_MODE=="walk":
            base = filter_walk_radius(base, START_LAT, START_LON, WALK_RADIUS_M)
        # Top-N 저장
        candidates_by_slot[slot] = base.head(20).copy()

    # 7) 일정 선택
    if TRAVEL_MODE=="walk":
        itin = select_itinerary_walk(candidates_by_slot, START_LAT, START_LON, LAMBDA_KM)
    else:
        # 대중교통: 거리 + (출발/도착 혼잡 평균) 페널티 반영
        itin = select_itinerary_transit(candidates_by_slot, emd_hourly, start, LAMBDA_KM, LAMBDA_CONG)

    # 8) 결과 엑셀 저장 (엑셀만)
    # 후보풀 병합
    pool_rows=[]
    for slot,dfc in candidates_by_slot.items():
        tmp=dfc.copy()
        if "name" not in tmp.columns and "poi_id" in tmp.columns:
            tmp["name"]=tmp["poi_id"]
        tmp.insert(0,"슬롯",slot)
        pool_rows.append(tmp)
    pool = pd.concat(pool_rows, ignore_index=True) if pool_rows else pd.DataFrame()

    params = pd.DataFrame({
        "param":["USER_QUERY","TRAVEL_MODE","SCHEDULE",
                 "START_LAT","START_LON","WALK_RADIUS_M",
                 "ALPHA","BETA","GAMMA","DELTA","LAMBDA_KM","LAMBDA_CONG","AVG_TRANSIT_KMPH","THEME_WEIGHTS"],
        "value":[USER_QUERY, TRAVEL_MODE, SCHEDULE,
                 START_LAT, START_LON, WALK_RADIUS_M,
                 ALPHA, BETA, GAMMA, DELTA, LAMBDA_KM, LAMBDA_CONG, AVG_TRANSIT_KMPH, str(THEME_WEIGHTS)]
    })

    with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
        params.to_excel(w, sheet_name="파라미터", index=False)
        keep_pool = [c for c in ["슬롯","poi_id","name","시도","시군구","읍면동","lat","lon",
                                 "관광지수_norm","인기도_norm","theme_score","혼잡도_norm",
                                 "final_score","시간대"] if c in pool.columns]
        pool[keep_pool].to_excel(w, sheet_name="후보풀", index=False)
        itin_cols = [c for c in ["시간대","poi_id","name","시도","시군구","읍면동","lat","lon",
                                 "관광지수_norm","인기도_norm","theme_score","혼잡도_norm",
                                 "final_score","이동거리_km","score_adj"] if c in itin.columns]
        itin[itin_cols].to_excel(w, sheet_name="일정표", index=False)

    print(f"[DONE] 저장: {OUT_XLSX}")

C:\Users\hyunj\AppData\Local\Temp\ipykernel_42388\648881247.py:315: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  poi=poi.groupby("시도", group_keys=False).apply(z01)
C:\Users\hyunj\AppData\Local\Temp\ipykernel_42388\648881247.py:326: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sagg=sagg.groupby(["시도","시군구"], group_keys=False).apply(mm)


ValueError: '시도' is both an index level and a column label, which is ambiguous.